# Recommender evaluation diagnostics

This notebook audits one completed batch-job run. It uses the uploaded test artifacts and rebuilds the complete candidate catalogue from the exact input CSV and uploaded model metadata. Run it from the repository root.

The input CSV must be the same catalogue used by the batch job. The notebook deliberately does not train, write to the database, or modify the evaluator.

In [1]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'song_recommendation').is_dir():
    raise RuntimeError('Open this notebook with the repository root as the working directory.')

# Set this to the same bucket URI supplied to the batch job.
GCS_BUCKET_URI = 'gs://YOUR_BUCKET/YOUR_PREFIX'
DOWNLOAD_FROM_GCS = True

# This must be the exact input catalogue used by the batch job.
DATA_CSV = PROJECT_ROOT / 'song_recommendation' / 'sample_data.csv'
ARTIFACT_DIR = PROJECT_ROOT / 'song_recommendation' / 'evaluation_results'

if not DATA_CSV.is_file():
    raise FileNotFoundError(f'Catalogue CSV not found: {DATA_CSV}')

RuntimeError: Open this notebook with the repository root as the working directory.

In [ ]:
# Download the artifacts produced by the batch job. This requires Google
# Application Default Credentials with object-read permission. Set
# DOWNLOAD_FROM_GCS = False if the files have already been downloaded.
ARTIFACT_FILENAMES = (
    'evaluation_true_test_vectors.npy',
    'evaluation_model_predicted_test_vectors.npy',
    'evaluation_top_10_predicted_test_vectors.npy',
    'evaluation_predictions.csv',
    'model_metadata.json',
)

if DOWNLOAD_FROM_GCS:
    from google.cloud import storage
    from song_recommendation.gcs_artifacts import parse_gcs_uri

    bucket_name, prefix = parse_gcs_uri(GCS_BUCKET_URI)
    bucket = storage.Client().bucket(bucket_name)
    ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
    for filename in ARTIFACT_FILENAMES:
        object_name = '/'.join(part for part in (prefix, filename) if part)
        bucket.blob(object_name).download_to_filename(ARTIFACT_DIR / filename)

missing = [name for name in ARTIFACT_FILENAMES if not (ARTIFACT_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f'Missing diagnostic artifacts: {missing}')

In [ ]:
import json

import numpy as np
import pandas as pd
from IPython.display import display

from song_recommendation.evaluation import evaluate_retrieval

true_vectors = np.load(ARTIFACT_DIR / 'evaluation_true_test_vectors.npy')
model_predicted_vectors = np.load(ARTIFACT_DIR / 'evaluation_model_predicted_test_vectors.npy')
top_retrieved_vectors = np.load(ARTIFACT_DIR / 'evaluation_top_10_predicted_test_vectors.npy')
query_details = pd.read_csv(ARTIFACT_DIR / 'evaluation_predictions.csv', dtype={'true_song_id': str})
with (ARTIFACT_DIR / 'model_metadata.json').open(encoding='utf-8') as handle:
    metadata = json.load(handle)

true_song_ids = query_details['true_song_id'].to_numpy(dtype=str)
if len(true_song_ids) != true_vectors.shape[0]:
    raise ValueError('The answer-key vector rows do not match evaluation_predictions.csv.')
if model_predicted_vectors.shape != true_vectors.shape:
    raise ValueError('Raw model predictions and answer-key vectors are not aligned.')
if top_retrieved_vectors.shape != (len(true_song_ids), 10, true_vectors.shape[1]):
    raise ValueError('Expected exactly ten retrieved vectors per test query.')

print(f'Loaded {len(true_song_ids):,} held-out queries with {true_vectors.shape[1]} features.')

In [ ]:
# Recreate the candidate vectors exactly as the training job did, using
# the persisted training-only preprocessing statistics and category levels.
NUMERIC_COLUMNS = (
    'dance', 'acoustic', 'aggressive', 'electronic', 'happy', 'party',
    'relaxed', 'sad', 'timbre', 'tonal', 'voice',
)

def rebuild_candidate_vectors(data_csv: Path, saved_metadata: dict) -> pd.DataFrame:
    data = pd.read_csv(data_csv, index_col='id')
    data.index = data.index.astype(str)
    data = data.loc[~data.index.duplicated(keep='first')].copy()
    required = set(NUMERIC_COLUMNS) | {'year'} | set(saved_metadata['category_levels'])
    missing_columns = sorted(required - set(data.columns))
    if missing_columns:
        raise ValueError(f'Catalogue is missing required columns: {missing_columns}')

    vectors = data.loc[:, NUMERIC_COLUMNS].astype(float).copy()
    vectors['year_std'] = (
        data['year'].astype(float) - float(saved_metadata['year_mean'])
    ) / float(saved_metadata['year_scale'])
    for column, levels in saved_metadata['category_levels'].items():
        encoded = pd.get_dummies(data[column].astype(str), prefix=column, dtype=float)
        expected = [f'{column}_{level}' for level in levels]
        vectors = pd.concat([vectors, encoded.reindex(columns=expected, fill_value=0.0)], axis=1)

    return vectors.reindex(columns=saved_metadata['target_columns'], fill_value=0.0).astype(np.float32)

candidate_frame = rebuild_candidate_vectors(DATA_CSV, metadata)
candidate_song_ids = candidate_frame.index.to_numpy(dtype=str)
candidate_vectors = candidate_frame.to_numpy()

if not np.isin(true_song_ids, candidate_song_ids).all():
    raise ValueError('The catalogue CSV does not contain every held-out song ID.')
np.testing.assert_allclose(
    candidate_frame.loc[true_song_ids].to_numpy(), true_vectors, rtol=1e-5, atol=1e-6
)
print(f'Rebuilt {len(candidate_song_ids):,} candidate vectors from {DATA_CSV.name}.')

In [ ]:
# 1. Answer-key retrieval test: every query is its true held-out vector.
answer_key_metrics, answer_key_details, _ = evaluate_retrieval(
    predicted_vectors=true_vectors,
    true_song_ids=true_song_ids,
    candidate_vectors=candidate_vectors,
    candidate_song_ids=candidate_song_ids,
)

score_summary = pd.DataFrame([{
    'top_1_accuracy': answer_key_metrics['top_1']['accuracy'],
    **{f'hit_rate_{k}': values['hit_rate'] for k, values in answer_key_metrics['retrieval'].items()},
}])
display(score_summary)

def report_vector_ties(vectors: np.ndarray, label: str) -> pd.DataFrame:
    # An exact duplicate gives FAISS no deterministic ID-level preference.
    _, inverse, counts = np.unique(vectors, axis=0, return_inverse=True, return_counts=True)
    candidate_positions = pd.Index(candidate_song_ids).get_indexer(true_song_ids)
    tied_rows = []
    for query_id, candidate_position in zip(true_song_ids, candidate_positions):
        group = inverse[candidate_position]
        if counts[group] > 1:
            tied_rows.append({
                'tie_type': label,
                'true_song_id': query_id,
                'n_identical_vectors': int(counts[group]),
                'song_ids_sharing_vector': ', '.join(candidate_song_ids[inverse == group]),
            })
    return pd.DataFrame(tied_rows)

raw_vector_ties = report_vector_ties(candidate_vectors, 'raw target vector')
print(f'Perfect top-1 answer-key score: {answer_key_metrics["top_1"]["accuracy"] == 1.0}')
print(f'Held-out queries with an identical target-vector tie: {len(raw_vector_ties):,}')
display(raw_vector_ties.head(20))
raw_vector_ties.to_csv(ARTIFACT_DIR / 'answer_key_identical_vector_ties.csv', index=False)

In [ ]:
# 2. Cosine similarity of each answer key against each of its ten retrieved vectors.
def l2_normalise(values: np.ndarray, axis: int) -> np.ndarray:
    norms = np.linalg.norm(values, axis=axis, keepdims=True)
    return values / np.maximum(norms, np.finfo(np.float32).eps)

cosine_by_rank = np.einsum(
    'nf,nkf->nk',
    l2_normalise(true_vectors, axis=1),
    l2_normalise(top_retrieved_vectors, axis=2),
)
cosine_by_query = pd.DataFrame(
    cosine_by_rank,
    columns=[f'prediction_{rank}_cosine_similarity' for rank in range(1, 11)],
)
cosine_by_query.insert(0, 'true_song_id', true_song_ids)
cosine_rank_averages = pd.DataFrame({
    'prediction_rank': range(1, 11),
    'mean_cosine_similarity': cosine_by_rank.mean(axis=0),
})
display(cosine_rank_averages)
cosine_by_query.to_csv(ARTIFACT_DIR / 'cosine_similarity_by_query.csv', index=False)
cosine_rank_averages.to_csv(ARTIFACT_DIR / 'cosine_similarity_rank_averages.csv', index=False)

In [ ]:
# 3. Per-feature errors use only the top-ranked retrieved vector, as requested.
top_1_predicted_vectors = top_retrieved_vectors[:, 0, :]
feature_names = metadata['target_columns']
if len(feature_names) != true_vectors.shape[1]:
    raise ValueError('Metadata feature names do not match the vector width.')

feature_rows = []
for feature_index, feature_name in enumerate(feature_names):
    actual = true_vectors[:, feature_index]
    predicted = top_1_predicted_vectors[:, feature_index]
    error = predicted - actual
    total_sum_of_squares = np.square(actual - actual.mean()).sum()
    r_squared = (
        np.nan if total_sum_of_squares == 0
        else 1 - np.square(error).sum() / total_sum_of_squares
    )
    correlation = (
        np.nan if np.std(actual) == 0 or np.std(predicted) == 0
        else np.corrcoef(actual, predicted)[0, 1]
    )
    feature_rows.append({
        'feature': feature_name,
        'mae': np.abs(error).mean(),
        'rmse': np.sqrt(np.square(error).mean()),
        'bias_predicted_minus_true': error.mean(),
        'r_squared': r_squared,
        'pearson_correlation': correlation,
    })

per_feature_metrics = pd.DataFrame(feature_rows)
display(per_feature_metrics)
per_feature_metrics.to_csv(ARTIFACT_DIR / 'top_1_per_feature_metrics.csv', index=False)